In [0]:
STORAGE_ACCOUNT   = "storageaccountfull"
BRONZE_CONTAINER  = "bronze"
BRONZE_PATH       = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

CATALOG           = "e-commerce"
SILVER_DB         = "silver"
TARGET_TABLE      = "dim_customers"
NATURAL_KEY       = "customer_id"

In [0]:


from pyspark.sql.functions import (
    col, trim, upper, lower, initcap, expr,
    when, lit, current_timestamp,
    monotonically_increasing_id, row_number, coalesce
)
from pyspark.sql.types import IntegerType, DateType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
print(" Imports ready ")

✅ Imports ready | Database ensured


In [0]:

df_bronze = (
    spark.read
    .format("parquet")
    .load(f"{BRONZE_PATH}/customers")
    .withColumn("source_file", col("_metadata.file_path"))
)

print(f" Bronze rows : {df_bronze.count():,}")
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

📥 Bronze rows : 50,014
root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- loyalty_member: string (nullable = true)
 |-- join_date: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file: string (nullable = false)

+-----------+---+------+--------------+----------+-------------+------------------------------------------------------------------------------------------------------------------------------------+
|customer_id|age|gender|loyalty_member|join_date |_rescued_data|source_file                                                                                                                         |
+-----------+---+------+--------------+----------+-------------+------------------------------------------------------------------------------------------------------------------------------------+
|C000001    |40 |Male  |1             |2025-05-21|NULL         |abfss://bronze@storag

In [0]:


df_clean = (
    df_bronze

    .filter(col(NATURAL_KEY).isNotNull())

    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("gender",      trim(col("gender")))

    .withColumn("customer_id",
        expr("CONCAT('C', LPAD(REGEXP_EXTRACT(customer_id, '[0-9]+', 0), 6, '0'))")
    )

    .withColumn("gender",
        when(upper(col("gender")).isin("M", "MALE"),    lit("Male"))
       .when(upper(col("gender")).isin("F", "FEMALE"),  lit("Female"))
       .otherwise(lit("Unknown"))
    )

    .withColumn("age",            col("age").cast(IntegerType()))
    .withColumn("loyalty_member", col("loyalty_member").cast(IntegerType()))
    .withColumn("join_date",
        coalesce(
            expr("try_to_date(join_date, 'yyyy-MM-dd')"),
            expr("try_to_date(join_date, 'dd-MM-yyyy')")
        )
    )

    .withColumn("age",
        when((col("age") < 0) | (col("age") > 120), lit(None))
       .otherwise(col("age"))
    )

    .withColumn("age_band",
        when(col("age") < 25,  lit("Under 25"))
       .when(col("age") < 35,  lit("25-34"))
       .when(col("age") < 45,  lit("35-44"))
       .when(col("age") < 55,  lit("45-54"))
       .when(col("age") < 65,  lit("55-64"))
       .otherwise(lit("65+"))
    )

    .withColumn("loyalty_label",
        when(col("loyalty_member") == 1, lit("Loyalty Member"))
       .otherwise(lit("Non-Member"))
    )
)

print(f" Clean rows : {df_clean.count():,}")

✅ Clean rows : 50,014


In [0]:


window_spec = Window.partitionBy(NATURAL_KEY).orderBy(col("join_date").desc())

df_deduped = (
    df_clean
    .withColumn("_rn", row_number().over(window_spec))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

print(f" After dedup : {df_deduped.count():,}")

📊 After dedup : 50,010


In [0]:


import warnings
warnings.filterwarnings("ignore", message=".*No Partition Defined for Window operation.*")


if spark.catalog.tableExists(f"{SILVER_DB}.{TARGET_TABLE}"):
    existing_sk = (
        spark.table(f"{SILVER_DB}.{TARGET_TABLE}")
        .select("customer_sk", NATURAL_KEY)
    )

    df_with_sk = (
        df_deduped
        .join(existing_sk, on=NATURAL_KEY, how="left")
    )

    max_sk = existing_sk.agg({"customer_sk": "max"}).collect()[0][0] or 0

    df_existing = df_with_sk.filter(col("customer_sk").isNotNull())
    df_new = (
        df_with_sk
        .filter(col("customer_sk").isNull())
        .drop("customer_sk")
        .withColumn("_row_id", monotonically_increasing_id())
    )

    window_new = Window.orderBy("_row_id")
    df_new = (
        df_new
        .withColumn("customer_sk", (row_number().over(window_new) + max_sk).cast(IntegerType()))
        .drop("_row_id")
    )

    df_with_sk = df_existing.unionByName(df_new)
    print(f"   Existing customers (SK reused) : {df_existing.count():,}")
    print(f"   New customers (SK assigned)    : {df_new.count():,}")

else:
    window_new = Window.orderBy(NATURAL_KEY)
    df_with_sk = (
        df_deduped
        .withColumn("customer_sk", row_number().over(window_new).cast(IntegerType()))
    )
    print(f"   First run — assigning SKs 1 → {df_with_sk.count():,}")

   Existing customers (SK reused) : 50,010
   New customers (SK assigned)    : 0


In [0]:
df_final = (
    df_with_sk
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("updated_at",     current_timestamp())
)

In [0]:

df_dim = df_final.select(
    col("customer_sk"),
    col("customer_id"),
    col("age"),
    col("age_band"),
    col("gender"),
    col("loyalty_member"),
    col("loyalty_label"),
    col("join_date"),
    col("ingestion_time"),
    col("source_file"),
    col("updated_at"),
)

print(" Final dim schema:")
df_dim.printSchema()
df_dim.show(5, truncate=False)

✅ Final dim schema:
root
 |-- customer_sk: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- age_band: string (nullable = false)
 |-- gender: string (nullable = false)
 |-- loyalty_member: integer (nullable = true)
 |-- loyalty_label: string (nullable = false)
 |-- join_date: date (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)
 |-- source_file: string (nullable = false)
 |-- updated_at: timestamp (nullable = false)

+-----------+-----------+---+--------+------+--------------+--------------+----------+-------------------------+------------------------------------------------------------------------------------------------------------------------------------+-------------------------+
|customer_sk|customer_id|age|age_band|gender|loyalty_member|loyalty_label |join_date |ingestion_time           |source_file                                                                                                        

In [0]:


if not spark.catalog.tableExists(f"{SILVER_DB}.{TARGET_TABLE}"):
    df_dim.repartition(4).write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_DB}.{TARGET_TABLE}")
    print(f" {SILVER_DB}.{TARGET_TABLE} created (4 partitions)")

else:
    df_dim.createOrReplaceTempView("dim_customers_updates")

    spark.sql(f"""
        MERGE INTO {SILVER_DB}.{TARGET_TABLE}  AS target
        USING dim_customers_updates            AS source
        ON target.customer_id = source.customer_id

        WHEN MATCHED THEN UPDATE SET
            target.age            = source.age,
            target.age_band       = source.age_band,
            target.gender         = source.gender,
            target.loyalty_member = source.loyalty_member,
            target.loyalty_label  = source.loyalty_label,
            target.join_date      = source.join_date,
            target.updated_at     = source.updated_at

        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f" MERGE complete → {SILVER_DB}.{TARGET_TABLE}")

✅ MERGE complete → silver.dim_customers


In [0]:

spark.sql(f"SELECT COUNT(*) AS total_rows FROM {SILVER_DB}.{TARGET_TABLE}").show()

spark.sql(f"""
    SELECT gender, age_band, loyalty_label, COUNT(*) AS cnt
    FROM {SILVER_DB}.{TARGET_TABLE}
    GROUP BY gender, age_band, loyalty_label
    ORDER BY gender, age_band
""").show(20)

spark.sql(f"SELECT * FROM {SILVER_DB}.{TARGET_TABLE} LIMIT 5").show(truncate=False)


+----------+
|total_rows|
+----------+
|     50010|
+----------+

+------+--------+--------------+----+
|gender|age_band| loyalty_label| cnt|
+------+--------+--------------+----+
|Female|   25-34|Loyalty Member|2285|
|Female|   25-34|    Non-Member|2307|
|Female|   35-44|Loyalty Member|2329|
|Female|   35-44|    Non-Member|2338|
|Female|   45-54|Loyalty Member|2295|
|Female|   45-54|    Non-Member|2318|
|Female|   55-64|Loyalty Member|2374|
|Female|   55-64|    Non-Member|2317|
|Female|     65+|    Non-Member|1400|
|Female|     65+|Loyalty Member|1430|
|Female|Under 25|    Non-Member|1693|
|Female|Under 25|Loyalty Member|1678|
|  Male|   25-34|Loyalty Member|2351|
|  Male|   25-34|    Non-Member|2289|
|  Male|   35-44|Loyalty Member|2411|
|  Male|   35-44|    Non-Member|2397|
|  Male|   45-54|Loyalty Member|2433|
|  Male|   45-54|    Non-Member|2376|
|  Male|   55-64|Loyalty Member|2419|
|  Male|   55-64|    Non-Member|2402|
+------+--------+--------------+----+
only showing top 20 ro

In [0]:
%python
spark.sql(f"SELECT COUNT(*) AS total_rows FROM {SILVER_DB}.{TARGET_TABLE}").show()

+----------+
|total_rows|
+----------+
|     50010|
+----------+

